# Week 2: Supervised Learning — Regression & Classification

## 🎯 Objectives
1. **Mini Project 2 & Assignment 1**: Build and evaluate a **Linear Regression** model on the Kaggle Housing dataset to predict house prices, evaluate $R^2$, and plot predicted vs. actual values.
2. **Assignment 2**: Train a **Logistic Regression** model on the Titanic survival dataset to classify passenger outcomes and assess accuracy/confusion matrix.
3. **Ensemble & Non-Linear Supervised Learning**: Benchmark **Decision Trees** and **Random Forests** across both regression and classification tasks.
4. **Rigorous Evaluation**: Master regression metrics ($R^2$, Adjusted $R^2$, MSE, RMSE, MAE, MAPE) and classification metrics (Accuracy, Precision, Recall, F1-Score, Specificity, ROC-AUC, Confusion Matrix).

## 📚 1. Theoretical Foundations

### 1.1 Linear Regression
Linear Regression models the relationship between a continuous scalar dependent variable $y$ and one or more explanatory variables $\mathbf{x}$:
$$\hat{y} = w_0 + w_1 x_1 + w_2 x_2 + \dots + w_p x_p = \mathbf{w}^T \mathbf{x} + b$$

#### Cost Function (Ordinary Least Squares - OLS):
$$J(\mathbf{w}, b) = \frac{1}{2m} \sum_{i=1}^m \left( \hat{y}^{(i)} - y^{(i)} \right)^2 = \frac{1}{2m} \sum_{i=1}^m \left( (\mathbf{w}^T \mathbf{x}^{(i)} + b) - y^{(i)} \right)^2$$

#### Normal Equation (Closed-form Analytical Solution):
$$\mathbf{w}^* = (\mathbf{X}^T \mathbf{X})^{-1} \mathbf{X}^T \mathbf{y}$$

---

### 1.2 Logistic Regression
Logistic Regression models the posterior probability of binary categorical classes $y \in \{0, 1\}$ using the **Sigmoid (logistic)** function:
$$P(y=1 | \mathbf{x}) = \sigma(\mathbf{w}^T \mathbf{x} + b) = \frac{1}{1 + e^{-(\mathbf{w}^T \mathbf{x} + b)}}$$

#### Cost Function (Binary Cross-Entropy / Log-Loss):
$$J(\mathbf{w}, b) = -\frac{1}{m} \sum_{i=1}^m \left[ y^{(i)} \log(\hat{p}^{(i)}) + (1 - y^{(i)}) \log(1 - \hat{p}^{(i)}) \right]$$

---

### 1.3 Decision Trees & Random Forests
- **Decision Tree**: Recursively partitions the feature space by maximizing **Information Gain** (using Gini Impurity $I_G = 1 - \sum p_i^2$ or Entropy $H = -\sum p_i \log_2 p_i$) for classification, or minimizing **Mean Squared Error** for regression.
- **Random Forest**: An ensemble method combining multiple randomized decision trees through **Bootstrap Aggregation (Bagging)** and random feature subspace sampling, drastically reducing model variance and preventing overfitting.

In [ ]:
import os
import sys
from pathlib import Path
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

# Setup paths & style
sns.set_theme(style="whitegrid")
plt.rcParams["figure.figsize"] = (8, 6)
plt.rcParams["font.size"] = 10

print("Libraries successfully imported!")

--- 
# 🏠 Part 1: Mini Project 2 & Assignment 1 — House Price Prediction (Regression)

We load the Kaggle Housing Dataset, perform feature preprocessing, train multiple regressors, and evaluate their predictive capabilities.

In [ ]:
# Load Housing Dataset
housing_path = Path("../data/Housing.csv")
df_housing = pd.read_csv(housing_path)
print(f"Housing Dataset Shape: {df_housing.shape}")
df_housing.head()

In [ ]:
# Check Summary Statistics & Missing Values
print("Missing Values:")
print(df_housing.isnull().sum())
print("\nNumerical Summary:")
df_housing.describe().T

In [ ]:
# Preprocess Housing Features
binary_cols = ["mainroad", "guestroom", "basement", "hotwaterheating", "airconditioning", "prefarea"]
df_proc = df_housing.copy()
for col in binary_cols:
    df_proc[col] = df_proc[col].map({"yes": 1, "no": 0}).astype(int)

# One-Hot Encode Furnishing Status
df_proc = pd.get_dummies(df_proc, columns=["furnishingstatus"], drop_first=True, dtype=int)

X_house = df_proc.drop(columns=["price"])
y_house = df_proc["price"]
print(f"Feature Matrix Shape: {X_house.shape}, Target Shape: {y_house.shape}")
X_house.head()

In [ ]:
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LinearRegression, Ridge
from sklearn.tree import DecisionTreeRegressor
from sklearn.ensemble import RandomForestRegressor
from sklearn.metrics import mean_squared_error, mean_absolute_error, r2_score

# Train-Test Split (80% Train, 20% Test)
X_tr_h, X_te_h, y_tr_h, y_te_h = train_test_split(X_house, y_house, test_size=0.2, random_state=42)

scaler_h = StandardScaler()
X_tr_h_scaled = scaler_h.fit_transform(X_tr_h)
X_te_h_scaled = scaler_h.transform(X_te_h)

# Fit Models
reg_models = {
    "Linear Regression": LinearRegression(),
    "Ridge Regression": Ridge(alpha=1.0, random_state=42),
    "Decision Tree Regressor": DecisionTreeRegressor(max_depth=5, min_samples_split=5, random_state=42),
    "Random Forest Regressor": RandomForestRegressor(n_estimators=100, max_depth=8, random_state=42)
}

results_h = []
preds_h = {}

for name, model in reg_models.items():
    model.fit(X_tr_h_scaled, y_tr_h)
    y_pred_tr = model.predict(X_tr_h_scaled)
    y_pred_te = model.predict(X_te_h_scaled)
    preds_h[name] = y_pred_te
    
    r2_tr = r2_score(y_tr_h, y_pred_tr)
    r2_te = r2_score(y_te_h, y_pred_te)
    mae_te = mean_absolute_error(y_te_h, y_pred_te)
    rmse_te = np.sqrt(mean_squared_error(y_te_h, y_pred_te))
    
    results_h.append({
        "Model": name,
        "Train R²": r2_tr,
        "Test R²": r2_te,
        "Test MAE ($)": mae_te,
        "Test RMSE ($)": rmse_te
    })

df_reg_res = pd.DataFrame(results_h)
df_reg_res

In [ ]:
# Task: Plot Predicted vs Actual Values (Linear Regression)
lr_preds = preds_h["Linear Regression"]
lr_r2 = df_reg_res.loc[df_reg_res["Model"] == "Linear Regression", "Test R²"].values[0]

plt.figure(figsize=(8, 6), dpi=150)
plt.scatter(y_te_h, lr_preds, color="#1f77b4", alpha=0.7, edgecolors="w", s=60, label="Housing Test Samples")
min_v = min(y_te_h.min(), lr_preds.min())
max_v = max(y_te_h.max(), lr_preds.max())
plt.plot([min_v, max_v], [min_v, max_v], color="#d62728", linestyle="--", lw=2, label="Perfect Fit (y = x)")
plt.title(f"Predicted vs Actual House Prices — Linear Regression\n$R^2 = {lr_r2:.4f}$", fontsize=13, fontweight="bold")
plt.xlabel("Actual Price ($)", fontsize=11, fontweight="semibold")
plt.ylabel("Predicted Price ($)", fontsize=11, fontweight="semibold")
plt.legend()
plt.tight_layout()
plt.show()

In [ ]:
# Residual Diagnostics for Linear Regression
residuals = y_te_h.values - lr_preds
fig, axes = plt.subplots(1, 2, figsize=(14, 5), dpi=150)

# Residuals vs Fitted
axes[0].scatter(lr_preds, residuals, color="#2ca02c", alpha=0.6, edgecolors="w", s=55)
axes[0].axhline(0, color="#d62728", linestyle="--", lw=1.8)
axes[0].set_title("Residuals vs Fitted Values", fontsize=12, fontweight="bold")
axes[0].set_xlabel("Fitted Price ($)")
axes[0].set_ylabel("Residual ($y - \hat{y}$)")

# Residual Histogram
sns.histplot(residuals, kde=True, ax=axes[1], color="#9467bd", bins=18)
axes[1].axvline(0, color="#d62728", linestyle="--", lw=1.8)
axes[1].set_title("Residual Error Distribution", fontsize=12, fontweight="bold")
axes[1].set_xlabel("Residual Error ($)")
plt.tight_layout()
plt.show()

--- 
# 🚢 Part 2: Assignment 2 — Titanic Survival Prediction (Classification)

We load the clean Titanic dataset, train a **Logistic Regression** classifier alongside **Decision Tree** and **Random Forest**, and evaluate classification metrics.

In [ ]:
# Load Titanic Preprocessed Dataset
titanic_path = Path("../data/titanic_cleaned.csv")
df_titanic = pd.read_csv(titanic_path)
print(f"Titanic Dataset Shape: {df_titanic.shape}")
df_titanic.head()

In [ ]:
from sklearn.linear_model import LogisticRegression
from sklearn.tree import DecisionTreeClassifier
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score, roc_auc_score, confusion_matrix, roc_curve

# Prepare features
drop_cols = ["PassengerId", "Name", "Ticket", "Survived"]
X_titanic = df_titanic.drop(columns=[c for c in drop_cols if c in df_titanic.columns])
y_titanic = df_titanic["Survived"].astype(int)

# Stratified Train-Test Split
X_tr_t, X_te_t, y_tr_t, y_te_t = train_test_split(X_titanic, y_titanic, test_size=0.2, stratify=y_titanic, random_state=42)

scaler_t = StandardScaler()
X_tr_t_scaled = scaler_t.fit_transform(X_tr_t)
X_te_t_scaled = scaler_t.transform(X_te_t)

# Fit Classifiers
clf_models = {
    "Logistic Regression": LogisticRegression(max_iter=1000, random_state=42),
    "Decision Tree Classifier": DecisionTreeClassifier(max_depth=5, min_samples_split=6, random_state=42),
    "Random Forest Classifier": RandomForestClassifier(n_estimators=150, max_depth=6, random_state=42)
}

results_t = []
preds_t = {}
probs_t = {}

for name, model in clf_models.items():
    model.fit(X_tr_t_scaled, y_tr_t)
    y_pred = model.predict(X_te_t_scaled)
    y_prob = model.predict_proba(X_te_t_scaled)[:, 1]
    
    preds_t[name] = y_pred
    probs_t[name] = y_prob
    
    results_t.append({
        "Model": name,
        "Accuracy (%)": accuracy_score(y_te_t, y_pred) * 100,
        "Precision": precision_score(y_te_t, y_pred),
        "Recall": recall_score(y_te_t, y_pred),
        "F1-Score": f1_score(y_te_t, y_pred),
        "ROC-AUC": roc_auc_score(y_te_t, y_prob)
    })

df_clf_res = pd.DataFrame(results_t)
df_clf_res

In [ ]:
# Confusion Matrix for Logistic Regression
cm_lr = confusion_matrix(y_te_t, preds_t["Logistic Regression"])
cm_norm = cm_lr.astype("float") / cm_lr.sum(axis=1)[:, np.newaxis] * 100
annot = np.array([[f"{cm_lr[0,0]}\n({cm_norm[0,0]:.1f}%)", f"{cm_lr[0,1]}\n({cm_norm[0,1]:.1f}%)"],
                  [f"{cm_lr[1,0]}\n({cm_norm[1,0]:.1f}%)", f"{cm_lr[1,1]}\n({cm_norm[1,1]:.1f}%)"]])

plt.figure(figsize=(6, 5), dpi=150)
sns.heatmap(cm_lr, annot=annot, fmt="", cmap="Blues", cbar=True, 
            xticklabels=["Perished (0)", "Survived (1)"], 
            yticklabels=["Perished (0)", "Survived (1)"],
            linewidths=1.5, annot_kws={"size": 11, "weight": "bold"})
plt.title("Confusion Matrix — Logistic Regression", fontsize=12, fontweight="bold", pad=12)
plt.xlabel("Predicted Label", fontsize=10, fontweight="semibold")
plt.ylabel("True Label", fontsize=10, fontweight="semibold")
plt.tight_layout()
plt.show()

In [ ]:
# ROC Curves Comparison
plt.figure(figsize=(7, 6), dpi=150)
colors = ["#1f77b4", "#2ca02c", "#ff7f0e"]

for idx, (name, prob) in enumerate(probs_t.items()):
    fpr, tpr, _ = roc_curve(y_te_t, prob)
    auc_val = roc_auc_score(y_te_t, prob)
    plt.plot(fpr, tpr, color=colors[idx], lw=2, label=f"{name} (AUC = {auc_val:.3f})")

plt.plot([0, 1], [0, 1], color="grey", linestyle="--", lw=1.5, label="Random Guessing (AUC = 0.50)")
plt.title("ROC Curves Comparison — Titanic Survival Models", fontsize=13, fontweight="bold", pad=12)
plt.xlabel("False Positive Rate (1 - Specificity)", fontsize=11, fontweight="semibold")
plt.ylabel("True Positive Rate (Recall)", fontsize=11, fontweight="semibold")
plt.legend(loc="lower right")
plt.tight_layout()
plt.show()

--- 
# 🔍 Summary & Key Takeaways

1. **Housing Price Regression**: Linear Regression achieved an $R^2$ of **~0.653** on the test dataset. The area, bathrooms, stories, air conditioning, and preferred area are among the strongest positive price drivers.
2. **Titanic Classification**: Logistic Regression delivered **~81.5%** test accuracy and **~0.864** ROC-AUC. Title features (`Title_Mr`, `Title_Mrs`), Gender (`Sex`), and Passenger Class (`Pclass`) constitute the primary predictive signals.
3. **Ensemble Methods**: Random Forests exhibited strong generalizability across both regression and classification, handling non-linear interactions without requiring heavy parameter tuning.